# Embeddings Statuti Normattiva

Genera gli embeddings dei tre statuti estratti da Normattiva:

- `codice_penale_normattiva.csv`
- `codice_civile_normattiva.csv`
- `codice_amministrativo_normattiva.csv`

Output (solo `.npy`, nessun `.pkl`):

- `src/data/embeddings/penale_normattiva_embeddings.npy`
- `src/data/embeddings/civile_normattiva_embeddings.npy`
- `src/data/embeddings/amministrativo_normattiva_embeddings.npy`


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH = 512
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

STATUTES_DIR = PROJECT_ROOT / "src/data/statutes"
EMBEDDINGS_DIR = PROJECT_ROOT / "src/data/embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "penale_normattiva": {
        "csv": STATUTES_DIR / "codice_penale_normattiva.csv",
        "title_col": "titolo",
        "text_col": "testo",
    },
    "civile_normattiva": {
        "csv": STATUTES_DIR / "codice_civile_normattiva.csv",
        "title_col": "article_title",
        "text_col": "article_text",
    },
    "amministrativo_normattiva": {
        "csv": STATUTES_DIR / "codice_amministrativo_normattiva.csv",
        "title_col": "titolo",
        "text_col": "contenuto",
    },
}

print(f"Device: {DEVICE}")
print(f"Project root: {PROJECT_ROOT}")


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    pooled = (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return pooled


def build_texts(df: pd.DataFrame, title_col: str, text_col: str) -> list[str]:
    titles = df[title_col].fillna("").astype(str)
    texts = df[text_col].fillna("").astype(str)
    merged = []
    for t, x in zip(titles, texts):
        merged.append(f"{t}. {x}".strip())
    return merged


def embed_texts(texts: list[str], tokenizer, model, batch_size: int = BATCH_SIZE) -> np.ndarray:
    vectors: list[np.ndarray] = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding", unit="batch"):
        batch = texts[i : i + batch_size]
        encoded = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            outputs = model(**encoded)
        pooled = mean_pooling(outputs, encoded["attention_mask"])
        vectors.append(pooled.cpu().numpy())
    if not vectors:
        return np.zeros((0, 768), dtype=np.float32)
    return np.vstack(vectors).astype(np.float32)


print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
model = AutoModel.from_pretrained(MODEL_NAME, local_files_only=True, trust_remote_code=True).to(DEVICE)
model.eval()

for dataset_name, cfg in DATASETS.items():
    csv_path = cfg["csv"]
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing CSV: {csv_path}")

    print()
    print("=" * 72)
    print(f"Dataset: {dataset_name}")
    print(f"CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    texts = build_texts(df, cfg["title_col"], cfg["text_col"])
    embeddings = embed_texts(texts, tokenizer=tokenizer, model=model)

    out_path = EMBEDDINGS_DIR / f"{dataset_name}_embeddings.npy"
    np.save(out_path, embeddings)

    print(f"Rows: {len(df)}")
    print(f"Embeddings shape: {embeddings.shape}")
    print(f"Saved: {out_path}")

print()
print("Done. No .pkl files generated.")
